In [3]:
# ============================================================
# FITNESSAI COACH — Powered by Google Gemini (Free API)
# ============================================================
# Requisiti: google-generativeai, pandas, ipywidgets, tabulate
# API Key gratuita: https://aistudio.google.com/app/apikey
# ============================================================

# ============================================================
# [1] INSTALLAZIONE DIPENDENZE
# ============================================================
from IPython.display import clear_output, display, HTML
print("⏳ Installazione dipendenze in corso...")
import subprocess
subprocess.run(["pip", "install", "-q",
    "google-generativeai", "pandas", "ipywidgets", "tabulate", "markdown"
], check=True)
clear_output()
print("✅ Dipendenze installate!")

# ============================================================
# [2] IMPORT
# ============================================================
import os, math, json, re, time
import pandas as pd
import ipywidgets as widgets
import google.generativeai as genai

# ============================================================
# [3] STILE HTML
# ============================================================
display(HTML("""
<style>
  @import url('https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Sans:wght@300;400;500&display=swap');
  .fit-header {
    background: linear-gradient(135deg, #0f2027, #1a3a4a, #0f4c3a);
    border-radius: 16px; padding: 32px 36px; margin-bottom: 24px;
    font-family: 'Syne', sans-serif;
  }
  .fit-header h1 { color: #e8f5e9; font-size: 2.2em; margin: 0 0 6px 0; letter-spacing: -1px; }
  .fit-header p  { color: #80cbc4; font-family: 'DM Sans', sans-serif; font-size: 1em; margin: 0; }
  .fit-section {
    background: #f8fffe; border-left: 4px solid #00897b;
    border-radius: 8px; padding: 14px 20px; margin: 16px 0 8px 0;
    font-family: 'Syne', sans-serif; font-size: 1.05em; color: #1a3a4a;
  }
  .fit-result {
    background: #e8f5e9; border-radius: 12px; padding: 20px 24px;
    margin: 12px 0; font-family: 'DM Sans', sans-serif; color: #1b5e20;
    border: 1px solid #a5d6a7;
  }
  .fit-ai {
    background: linear-gradient(135deg, #e0f7fa, #e8f5e9);
    border-radius: 12px; padding: 20px 24px; margin: 12px 0;
    font-family: 'DM Sans', sans-serif; color: #004d40;
    border: 1px solid #80cbc4; white-space: pre-wrap; line-height: 1.7;
  }
  .fit-warn {
    background: #fff8e1; border-left: 4px solid #ffc107;
    border-radius: 8px; padding: 12px 18px; margin: 8px 0;
    font-family: 'DM Sans', sans-serif; font-size: 0.9em; color: #5d4037;
  }
</style>
"""))

display(HTML("""
<div class="fit-header">
  <h1>🏋️ FitnessAI Coach</h1>
  <p>Piano personalizzato di dieta e allenamento — Powered by Google Gemini (free tier)</p>
</div>
"""))

# ============================================================
# [4] API KEY GEMINI
# ============================================================
display(HTML('<div class="fit-section">🔑 Configurazione API Key Gemini</div>'))
display(HTML('<div class="fit-warn">Ottieni la tua API key gratuita su <a href="https://aistudio.google.com/app/apikey" target="_blank">aistudio.google.com</a> (account Google, nessuna carta richiesta)</div>'))

try:
    from google.colab import userdata
    GEMINI_KEY = userdata.get('GEMINI_API_KEY')
    if GEMINI_KEY:
        print("✅ API Key Gemini caricata dai Secrets di Colab.")
    else:
        raise ValueError("Secret vuoto")
except:
    from getpass import getpass
    GEMINI_KEY = getpass("🔑 Incolla la tua Gemini API Key (da aistudio.google.com): ")

genai.configure(api_key=GEMINI_KEY)
# gemini-2.0-flash è stato ritirato il 3 marzo 2026.
# gemini-2.5-flash-lite è il modello free tier più generoso (15 RPM, 1000 RPD).
model = genai.GenerativeModel("gemini-2.5-flash-lite")
print("✅ Gemini 2.5 Flash-Lite pronto! (free tier: 15 req/min, 1000 req/giorno)")

# ---- Helper con retry automatico per rate limit 429 ----
def gemini_call(prompt, max_retries=4):
    """Chiama Gemini con retry automatico in caso di 429 (rate limit)."""
    for attempt in range(max_retries):
        try:
            risposta = model.generate_content(prompt)
            return risposta.text
        except Exception as e:
            msg = str(e)
            if "429" in msg:
                # Estrae i secondi di attesa suggeriti dal messaggio, altrimenti usa backoff
                import re as _re
                match = _re.search(r'retry in (\d+)', msg)
                wait = int(match.group(1)) + 2 if match else (15 * (attempt + 1))
                wait = min(wait, 65)  # mai più di 65s
                print(f"⏳ Rate limit raggiunto. Attendo {wait}s e riprovo (tentativo {attempt+1}/{max_retries})...")
                time.sleep(wait)
            else:
                raise  # errore diverso, rilancia subito
    raise RuntimeError(f"Gemini non ha risposto dopo {max_retries} tentativi (429 persistente).")

# ============================================================
# [5] CALCOLI TDEE REALI
# ============================================================
def calcola_tdee(sesso, eta, altezza_cm, peso_kg, livello_attivita):
    """Formula Mifflin-St Jeor (standard clinico)"""
    if sesso == 'M':
        bmr = 10 * peso_kg + 6.25 * altezza_cm - 5 * eta + 5
    else:
        bmr = 10 * peso_kg + 6.25 * altezza_cm - 5 * eta - 161

    moltiplicatori = {
        "Sedentario":              1.2,
        "Leggermente attivo":      1.375,
        "Moderatamente attivo":    1.55,
        "Molto attivo":            1.725,
        "Estremamente attivo":     1.9,
    }
    fattore = moltiplicatori.get(livello_attivita, 1.375)
    tdee = bmr * fattore
    return round(bmr), round(tdee)

def calcola_macros(tdee, obiettivo, peso_kg):
    """Calcola proteine, carboidrati e grassi in base all'obiettivo"""
    if obiettivo == "Perdita di peso":
        kcal_target = tdee - 400          # deficit moderato
        prot_g  = round(peso_kg * 2.0)   # alto apporto proteico per preservare muscolo
        grassi_g = round(kcal_target * 0.28 / 9)
        carb_g  = round((kcal_target - prot_g * 4 - grassi_g * 9) / 4)
    elif obiettivo == "Aumento massa muscolare":
        kcal_target = tdee + 300
        prot_g  = round(peso_kg * 2.2)
        grassi_g = round(kcal_target * 0.25 / 9)
        carb_g  = round((kcal_target - prot_g * 4 - grassi_g * 9) / 4)
    elif obiettivo == "Tonificazione":
        kcal_target = tdee - 150
        prot_g  = round(peso_kg * 1.9)
        grassi_g = round(kcal_target * 0.27 / 9)
        carb_g  = round((kcal_target - prot_g * 4 - grassi_g * 9) / 4)
    else:  # Mantenimento
        kcal_target = tdee
        prot_g  = round(peso_kg * 1.6)
        grassi_g = round(kcal_target * 0.30 / 9)
        carb_g  = round((kcal_target - prot_g * 4 - grassi_g * 9) / 4)

    carb_g = max(carb_g, 50)  # minimo fisiologico
    return kcal_target, prot_g, carb_g, grassi_g

# ============================================================
# [6] FORM INTERATTIVO
# ============================================================
display(HTML('<div class="fit-section">📋 Il tuo profilo</div>'))

w_nome    = widgets.Text(value='Mario', description='Nome:', style={'description_width':'130px'}, layout=widgets.Layout(width='350px'))
w_eta     = widgets.IntSlider(value=30, min=14, max=80, description='Età:', style={'description_width':'130px'}, layout=widgets.Layout(width='420px'))
w_sesso   = widgets.RadioButtons(options=['M','F'], value='M', description='Sesso:', style={'description_width':'130px'})
w_altezza = widgets.IntSlider(value=175, min=140, max=220, description='Altezza (cm):', style={'description_width':'130px'}, layout=widgets.Layout(width='420px'))
w_peso    = widgets.FloatSlider(value=82.0, min=40, max=200, step=0.5, description='Peso attuale (kg):', style={'description_width':'130px'}, layout=widgets.Layout(width='420px'))
w_peso_t  = widgets.FloatSlider(value=74.0, min=40, max=200, step=0.5, description='Peso obiettivo (kg):', style={'description_width':'130px'}, layout=widgets.Layout(width='420px'))

display(HTML('<div class="fit-section">🎯 Obiettivo e allenamento</div>'))

w_obiettivo = widgets.Dropdown(
    options=['Perdita di peso','Aumento massa muscolare','Tonificazione','Mantenimento'],
    value='Perdita di peso', description='Obiettivo:', style={'description_width':'130px'}, layout=widgets.Layout(width='380px'))
w_durata   = widgets.Dropdown(
    options=['4 settimane','8 settimane','3 mesi','6 mesi'],
    value='3 mesi', description='Durata piano:', style={'description_width':'130px'}, layout=widgets.Layout(width='380px'))
w_attivita = widgets.Dropdown(
    options=['Sedentario','Leggermente attivo','Moderatamente attivo','Molto attivo','Estremamente attivo'],
    value='Leggermente attivo', description='Livello attività:', style={'description_width':'130px'}, layout=widgets.Layout(width='380px'))
w_tipo_all = widgets.SelectMultiple(
    options=['Palestra (pesi)','Calisthenics','Running/Cardio','HIIT','Yoga/Pilates','Sport di squadra'],
    value=['Palestra (pesi)'], description='Tipo allenamento:', style={'description_width':'130px'}, layout=widgets.Layout(width='380px', height='130px'))
w_giorni   = widgets.IntSlider(value=4, min=1, max=7, description='Giorni/settimana:', style={'description_width':'130px'}, layout=widgets.Layout(width='420px'))
w_minuti   = widgets.IntSlider(value=60, min=20, max=120, step=5, description='Minuti/sessione:', style={'description_width':'130px'}, layout=widgets.Layout(width='420px'))
w_luogo    = widgets.Dropdown(
    options=['Palestra attrezzata','Solo corpo libero','Casa con attrezzi base','Esterno/parco'],
    value='Palestra attrezzata', description='Dove ti alleni:', style={'description_width':'130px'}, layout=widgets.Layout(width='380px'))
w_livello  = widgets.Dropdown(
    options=['Principiante (< 6 mesi)','Intermedio (6 mesi - 2 anni)','Avanzato (> 2 anni)'],
    value='Principiante (< 6 mesi)', description='Esperienza:', style={'description_width':'130px'}, layout=widgets.Layout(width='380px'))

display(HTML('<div class="fit-section">🥗 Alimentazione</div>'))

w_dieta  = widgets.Dropdown(
    options=['Onnivoro','Vegetariano','Vegano','Mediterraneo','Low-carb/Keto'],
    value='Mediterraneo', description='Stile alimentare:', style={'description_width':'130px'}, layout=widgets.Layout(width='380px'))
w_pasti  = widgets.Dropdown(
    options=['2 pasti','3 pasti','4 pasti','5 pasti e spuntini'],
    value='3 pasti', description='N° pasti al giorno:', style={'description_width':'130px'}, layout=widgets.Layout(width='380px'))
w_intoll = widgets.Textarea(
    value='nessuna', description='Intolleranze/allergie:', style={'description_width':'130px'},
    layout=widgets.Layout(width='380px', height='70px'))
w_patol  = widgets.Textarea(
    value='nessuna', description='Condizioni mediche:', style={'description_width':'130px'},
    layout=widgets.Layout(width='380px', height='70px'))
w_note   = widgets.Textarea(
    value='', placeholder='Es: non mi piace il pesce, lavoro di notte, ho poco tempo per cucinare...',
    description='Note libere:', style={'description_width':'130px'},
    layout=widgets.Layout(width='480px', height='80px'))

for w in [w_nome, w_eta, w_sesso, w_altezza, w_peso, w_peso_t,
          w_obiettivo, w_durata, w_attivita, w_tipo_all, w_giorni, w_minuti, w_luogo, w_livello,
          w_dieta, w_pasti, w_intoll, w_patol, w_note]:
    display(w)

# ============================================================
# [7] PULSANTE E GENERAZIONE
# ============================================================
btn = widgets.Button(
    description="🚀 Genera il mio piano personalizzato",
    button_style='success',
    layout=widgets.Layout(width='400px', height='54px', margin='24px 0 0 0')
)
out = widgets.Output()
display(btn, out)

def genera_piano(b):
    with out:
        clear_output()

        # --- Raccolta dati ---
        nome     = w_nome.value.strip() or "Atleta"
        eta      = w_eta.value
        sesso    = w_sesso.value
        altezza  = w_altezza.value
        peso     = w_peso.value
        peso_t   = w_peso_t.value
        obiettivo= w_obiettivo.value
        durata   = w_durata.value
        attivita = w_attivita.value
        tipo_all = list(w_tipo_all.value)
        giorni   = w_giorni.value
        minuti   = w_minuti.value
        luogo    = w_luogo.value
        livello  = w_livello.value
        dieta    = w_dieta.value
        pasti    = w_pasti.value
        intoll   = w_intoll.value.strip()
        patol    = w_patol.value.strip()
        note     = w_note.value.strip()

        # Validazione base
        if obiettivo == "Perdita di peso" and peso_t >= peso:
            display(HTML('<div class="fit-warn">⚠️ Il peso obiettivo dovrebbe essere inferiore al peso attuale per "Perdita di peso".</div>'))
        if obiettivo == "Aumento massa muscolare" and peso_t <= peso:
            display(HTML('<div class="fit-warn">⚠️ Per aumentare massa il peso obiettivo dovrebbe essere superiore.</div>'))

        # --- TDEE e Macros ---
        bmr, tdee = calcola_tdee(sesso, eta, altezza, peso, attivita)
        kcal_target, prot_g, carb_g, grassi_g = calcola_macros(tdee, obiettivo, peso)
        imc = round(peso / (altezza/100)**2, 1)

        display(HTML(f"""
        <div class="fit-result">
          <b>📊 Analisi metabolica per {nome}</b><br><br>
          🔥 <b>BMR (metabolismo a riposo):</b> {bmr} kcal/giorno<br>
          ⚡ <b>TDEE (fabbisogno reale):</b> {tdee} kcal/giorno<br>
          🎯 <b>Calorie obiettivo:</b> {kcal_target} kcal/giorno<br>
          📐 <b>IMC:</b> {imc} {"(sottopeso)" if imc<18.5 else "(normopeso)" if imc<25 else "(sovrappeso)" if imc<30 else "(obesità)"}<br><br>
          <b>Macronutrienti giornalieri:</b><br>
          🥩 Proteine: <b>{prot_g}g</b> ({prot_g*4} kcal) &nbsp;|&nbsp;
          🍞 Carboidrati: <b>{carb_g}g</b> ({carb_g*4} kcal) &nbsp;|&nbsp;
          🫒 Grassi: <b>{grassi_g}g</b> ({grassi_g*9} kcal)
        </div>
        """))

        # --- PROMPT DIETA ---
        print("🥗 Generazione piano dietetico con Gemini...")

        prompt_dieta = f"""Sei un nutrizionista sportivo professionista certificato.
Crea un piano alimentare settimanale COMPLETO e DETTAGLIATO per questo utente:

PROFILO:
- Nome: {nome}, {eta} anni, sesso: {sesso}
- Peso: {peso} kg → obiettivo: {peso_t} kg
- Altezza: {altezza} cm, IMC: {imc}
- Obiettivo: {obiettivo} in {durata}
- Condizioni mediche: {patol}
- Intolleranze/allergie: {intoll}

PIANO ALIMENTARE:
- Calorie target: {kcal_target} kcal/giorno
- Proteine: {prot_g}g | Carboidrati: {carb_g}g | Grassi: {grassi_g}g
- Stile alimentare: {dieta}
- Numero pasti: {pasti}
- Note particolari: {note if note else "nessuna"}

ISTRUZIONI:
1. Crea un piano per tutti e 7 i giorni della settimana (Lunedì-Domenica)
2. Per ogni giorno elenca TUTTI i pasti con:
   - Alimenti specifici con grammature precise (es: "150g petto di pollo", "80g riso basmati crudo")
   - Metodo di cottura consigliato
   - Calorie e macros approssimativi per ogni pasto
3. Includi spuntini se appropriato al numero di pasti scelto
4. Varia i pasti per evitare monotonia
5. Aggiungi una sezione "LISTA DELLA SPESA SETTIMANALE" organizzata per categoria
6. Aggiungi una sezione "CONSIGLI PRATICI" per preparazione e meal prep
7. Rispetta le intolleranze e lo stile alimentare in modo RIGOROSO
8. Usa un linguaggio chiaro e pratico, in italiano

Formatta in modo leggibile con emoji e sezioni ben separate."""

        try:
            testo_dieta = gemini_call(prompt_dieta)
            display(HTML('<div class="fit-section">🥗 Piano Dietetico Settimanale</div>'))
            display(HTML(f'<div class="fit-ai">{testo_dieta}</div>'))

            # Salva CSV semplice
            with open('/content/piano_dieta.txt', 'w', encoding='utf-8') as f:
                f.write(f"PIANO DIETETICO — {nome}\n")
                f.write(f"Calorie: {kcal_target} kcal | P:{prot_g}g C:{carb_g}g G:{grassi_g}g\n\n")
                f.write(testo_dieta)
            print("💾 Salvato in /content/piano_dieta.txt")

        except Exception as e:
            display(HTML(f'<div class="fit-warn">❌ Errore Gemini (dieta): {e}</div>'))
            testo_dieta = "Errore nella generazione"

        # --- PROMPT ALLENAMENTO ---
        print("\n🏋️ Generazione scheda allenamento con Gemini...")

        prompt_workout = f"""Sei un personal trainer certificato con 15 anni di esperienza.
Crea una scheda di allenamento COMPLETA e PROFESSIONALE per questo utente:

PROFILO ATLETA:
- Nome: {nome}, {eta} anni, sesso: {sesso}
- Peso: {peso} kg, altezza: {altezza} cm
- Obiettivo: {obiettivo} in {durata}
- Livello esperienza: {livello}
- Tipo di allenamento preferito: {", ".join(tipo_all)}
- Luogo di allenamento: {luogo}
- Giorni disponibili: {giorni} giorni/settimana
- Durata sessione: {minuti} minuti
- Condizioni mediche da considerare: {patol}

ISTRUZIONI:
1. Crea una scheda per {giorni} giorni di allenamento con i giorni di riposo indicati
2. Per OGNI esercizio specifica:
   - Nome esercizio (italiano e inglese)
   - Serie x ripetizioni (o tempo per esercizi isometrici)
   - Peso suggerito o % del massimale (se appropriato)
   - Tempo di recupero tra le serie
   - Muscoli coinvolti (primari e secondari)
   - Note tecniche sulla corretta esecuzione
3. Struttura ogni sessione con:
   - Riscaldamento (5-10 min)
   - Parte principale (esercizi)
   - Defaticamento e stretching
4. Adatta gli esercizi al luogo di allenamento: {luogo}
5. Adatta l'intensità al livello: {livello}
6. Includi una sezione "PROGRESSIONE NEL TEMPO" con come aumentare intensità nelle settimane
7. Aggiungi una sezione "CONSIGLI DI RECUPERO" (sonno, stretching, integrazione se utile)
8. Usa un linguaggio tecnico ma comprensibile, in italiano

Formatta in modo chiaro con emoji, tabelle testuali e sezioni ben separate."""

        try:
            testo_workout = gemini_call(prompt_workout)
            display(HTML('<div class="fit-section">🏋️ Scheda di Allenamento</div>'))
            display(HTML(f'<div class="fit-ai">{testo_workout}</div>'))

            with open('/content/scheda_allenamento.txt', 'w', encoding='utf-8') as f:
                f.write(f"SCHEDA ALLENAMENTO — {nome}\n")
                f.write(f"Obiettivo: {obiettivo} | Livello: {livello} | {giorni}g/sett x {minuti}min\n\n")
                f.write(testo_workout)
            print("💾 Salvata in /content/scheda_allenamento.txt")

        except Exception as e:
            display(HTML(f'<div class="fit-warn">❌ Errore Gemini (workout): {e}</div>'))
            testo_workout = "Errore nella generazione"

        # --- CONSIGLI PERSONALIZZATI ---
        print("\n🧠 Generazione consigli personalizzati...")

        prompt_coach = f"""Sei un coach di wellness e performance. Dai consigli personalizzati, pratici e motivazionali a {nome}.

CONTESTO:
- Obiettivo: {obiettivo} in {durata}
- Calorie: {kcal_target} kcal | Proteine: {prot_g}g
- Allenamento: {giorni}g/sett, {minuti}min, {", ".join(tipo_all)}
- Note personali: {note if note else "nessuna"}

Fornisci:
1. I 3 pilastri fondamentali per raggiungere l'obiettivo di {nome}
2. Le 3 trappole comuni da evitare per chi ha il suo profilo
3. Una strategia per i momenti difficili (mancanza di motivazione)
4. Consigli pratici su sonno e recupero
5. Un messaggio motivazionale finale personalizzato

Sii diretto, concreto, evita frasi banali. In italiano."""

        try:
            testo_coach = gemini_call(prompt_coach)
            display(HTML('<div class="fit-section">🧠 Consigli del Coach</div>'))
            display(HTML(f'<div class="fit-ai">{testo_coach}</div>'))
        except Exception as e:
            display(HTML(f'<div class="fit-warn">❌ Errore Gemini (coach): {e}</div>'))

        # --- REPORT HTML COMPLETO ---
        html_report = f"""<!DOCTYPE html>
<html lang="it">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>Piano FitnessAI — {nome}</title>
<style>
  @import url('https://fonts.googleapis.com/css2?family=Syne:wght@700;800&family=DM+Sans:wght@300;400;500&display=swap');
  * {{ box-sizing: border-box; margin: 0; padding: 0; }}
  body {{ font-family: 'DM Sans', sans-serif; background: #f0faf8; color: #1a3a4a; padding: 40px 20px; }}
  .container {{ max-width: 860px; margin: 0 auto; }}
  header {{ background: linear-gradient(135deg, #0f2027, #1a3a4a, #0f4c3a); border-radius: 16px;
             padding: 40px; margin-bottom: 32px; color: #e8f5e9; }}
  header h1 {{ font-family: 'Syne', sans-serif; font-size: 2.4em; letter-spacing: -1px; margin-bottom: 8px; }}
  header p  {{ color: #80cbc4; font-size: 1.05em; }}
  .card {{ background: white; border-radius: 12px; padding: 28px 32px; margin-bottom: 24px;
            box-shadow: 0 2px 12px rgba(0,105,92,0.08); border-top: 4px solid #00897b; }}
  .card h2 {{ font-family: 'Syne', sans-serif; font-size: 1.4em; color: #004d40; margin-bottom: 16px; }}
  .macros {{ display: flex; gap: 16px; flex-wrap: wrap; margin-top: 12px; }}
  .macro-badge {{ background: #e8f5e9; border-radius: 8px; padding: 10px 16px;
                  font-weight: 500; color: #1b5e20; font-size: 0.95em; }}
  .content {{ white-space: pre-wrap; line-height: 1.8; font-size: 0.95em; color: #2d4a3e; }}
  footer {{ text-align: center; color: #80cbc4; font-size: 0.85em; margin-top: 40px; padding: 20px; }}
</style>
</head>
<body>
<div class="container">
  <header>
    <h1>🏋️ Piano FitnessAI per {nome}</h1>
    <p>Generato il {pd.Timestamp.now().strftime('%d/%m/%Y')} — Obiettivo: {obiettivo} in {durata}</p>
  </header>

  <div class="card">
    <h2>📊 Analisi Metabolica</h2>
    <p><strong>BMR (metabolismo basale):</strong> {bmr} kcal/giorno</p>
    <p><strong>TDEE (fabbisogno reale):</strong> {tdee} kcal/giorno</p>
    <p><strong>Calorie obiettivo:</strong> {kcal_target} kcal/giorno</p>
    <p><strong>IMC:</strong> {imc}</p>
    <div class="macros">
      <div class="macro-badge">🥩 Proteine: {prot_g}g</div>
      <div class="macro-badge">🍞 Carboidrati: {carb_g}g</div>
      <div class="macro-badge">🫒 Grassi: {grassi_g}g</div>
    </div>
  </div>

  <div class="card">
    <h2>🥗 Piano Dietetico Settimanale</h2>
    <div class="content">{testo_dieta}</div>
  </div>

  <div class="card">
    <h2>🏋️ Scheda di Allenamento</h2>
    <div class="content">{testo_workout}</div>
  </div>

  <footer>Piano generato da FitnessAI Coach · Powered by Google Gemini 2.0 Flash</footer>
</div>
</body>
</html>"""

        with open('/content/piano_fitness_completo.html', 'w', encoding='utf-8') as f:
            f.write(html_report)

        display(HTML("""
        <div class="fit-result">
          <b>✅ Piano completato! File salvati:</b><br>
          📄 <code>/content/piano_dieta.txt</code><br>
          🏋️ <code>/content/scheda_allenamento.txt</code><br>
          🌐 <code>/content/piano_fitness_completo.html</code> (report completo scaricabile)<br><br>
          Per scaricarli: pannello file a sinistra → tasto destro → Download
        </div>
        """))

btn.on_click(genera_piano)

✅ Dipendenze installate!


🔑 Incolla la tua Gemini API Key (da aistudio.google.com): ··········
✅ Gemini 2.5 Flash-Lite pronto! (free tier: 15 req/min, 1000 req/giorno)


Text(value='Mario', description='Nome:', layout=Layout(width='350px'), style=DescriptionStyle(description_widt…

IntSlider(value=30, description='Età:', layout=Layout(width='420px'), max=80, min=14, style=SliderStyle(descri…

RadioButtons(description='Sesso:', options=('M', 'F'), style=DescriptionStyle(description_width='130px'), valu…

...

Textarea(value='nessuna', description='Intolleranze/allergie:', layout=Layout(height='70px', width='380px'), s…

Textarea(value='nessuna', description='Condizioni mediche:', layout=Layout(height='70px', width='380px'), styl…

Textarea(value='', description='Note libere:', layout=Layout(height='80px', width='480px'), placeholder='Es: n…

Button(button_style='success', description='🚀 Genera il mio piano personalizzato', layout=Layout(height='54px'…

Output()